In [20]:
# Cell - 1
# If running in a fresh environment, uncomment:
# %pip install pandas tqdm langdetect --quiet

import json
import logging
import random
import re
import uuid
import xml.etree.ElementTree as ET
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Optional

import pandas as pd
from tqdm.auto import tqdm

try:
    from langdetect import detect, DetectorFactory
    DetectorFactory.seed = 0
    LANGDETECT_AVAILABLE = True
except ImportError:
    LANGDETECT_AVAILABLE = False

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("bpmn_pipeline")

In [48]:
# Cell - 2 
@dataclass
class Config:
    # --- Input ---
    # Accepts a single CSV path or a directory containing many CSV shards
    # (e.g. SAP-SAM's 0.csv, 10000.csv, 20000.csv, ...).
    input_path: Path = Path("./data")
    csv_glob: str = "*.csv"
    csv_sep: str = "\t"          # SAP-SAM export is tab-separated
    csv_encoding: str = "utf-8"
    chunksize: int = 2_000       # rows per chunk when streaming large files

    # --- Filtering ---
    target_language: str = "English"
    required_stencilset_substring: str = "bpmn2.0"
    # Only keep genuine flow-chart process diagrams, not choreography/
    # conversation/collaboration variants, per the SAP-SAM paper's
    # recommendation to treat these as distinct sub-populations.
    excluded_name_markers: tuple = ("choreography", "conversation")
    min_tasks: int = 2           # drop trivial/empty diagrams
    max_tasks: int = 200         # drop pathological outliers

    # --- Sampling ---
    sample_size: int = 3000
    random_seed: int = 42

    # --- Split ---
    train_ratio: float = 0.8

    # --- Output ---
    output_path: Path = Path("./data/processed")
    train_dirname: str = "train"
    eval_dirname: str = "eval"
    metadata_filename: str = "metadata.json"
    stats_filename: str = "stats_report.json"

    # --- Synthetic data generation ---
    synth_seed: int = 7
    default_currency: str = "USD"
    hourly_rate_range: tuple = (15, 120)
    process_time_range_min: tuple = (5, 240)   
    rework_time_fraction_range: tuple = (0.05, 0.25)  
    default_job_titles: tuple = (
        "Process Owner", "Department Manager", "Analyst",
        "Coordinator", "Specialist", "Clerk", "Supervisor",
    )
    default_org_name: str = "Synthetic Org"
    default_process_category_id: int = 1
    default_process_category_name: str = "Uncategorized"


CONFIG = Config()

CONFIG.input_path = Path(r"C:\Users\yousu\Downloads\SAP\sap_sam_2022\data")
CONFIG.sample_size = 3000

print("input_path:", CONFIG.input_path)
print("sample_size:", CONFIG.sample_size)

(CONFIG.output_path / CONFIG.train_dirname).mkdir(parents=True, exist_ok=True)
(CONFIG.output_path / CONFIG.eval_dirname).mkdir(parents=True, exist_ok=True)

CONFIG.csv_sep = ","
print("csv_sep updated to:", repr(CONFIG.csv_sep))

random.seed(CONFIG.random_seed)
logger.info("Configuration loaded. Output directory: %s", CONFIG.output_path.resolve())

02:45:37 | INFO     | Configuration loaded. Output directory: C:\Users\yousu\Downloads\SAP\sap_sam_2022\data\processed


input_path: C:\Users\yousu\Downloads\SAP\sap_sam_2022\data
sample_size: 3000
csv_sep updated to: ','


In [26]:
# Cell - 3
REQUIRED_COLUMNS = [
    "Revision ID", "Model ID", "Organization ID", "Datetime",
    "Model JSON", "Description", "Name", "Type", "Namespace",
]


def _is_valid_bpmn_json(raw_json: str, cfg: Config) -> Optional[dict]:
    '''Parse the Model JSON string and return the dict iff it's a genuine
    BPMN 2.0 process diagram matching the config's filters. Returns None
    (never raises) if the row should be dropped.'''
    if not raw_json or not isinstance(raw_json, str):
        return None
    try:
        model = json.loads(raw_json)
    except (json.JSONDecodeError, TypeError):
        return None

    stencilset = model.get("stencilset", {}) or {}
    namespace = (stencilset.get("namespace") or "") + (stencilset.get("url") or "")
    if cfg.required_stencilset_substring not in namespace.lower():
        return None

    if model.get("stencil", {}).get("id") != "BPMNDiagram":
        return None

    return model


def _extract_language(model: dict, name: str, description: str, cfg: Config) -> Optional[str]:
    '''Cross-check the diagram's declared language property against the
    actual text content, since the declared property reflects the
    workspace's UI locale at creation time, not necessarily the language
    the modeler actually typed labels in.'''
    declared = (model.get("properties", {}) or {}).get("language")
    text = f"{name or ''} {description or ''}".strip()

    if not LANGDETECT_AVAILABLE or len(text) < 3:
        # No way to verify — fall back to trusting the declared field alone
        return declared

    try:
        detected_code = detect(text)
    except Exception:
        return declared

    detected_lang = "English" if detected_code == "en" else detected_code

    # Only trust "English" if detection agrees; for non-English declared
    # values we still return the detected language so filtering is accurate.
    if declared == cfg.target_language:
        return detected_lang if detected_code == "en" else detected_code
    return detected_lang


print("✅ Cell 6b (patch) OK — _extract_language now verifies against actual text")


def _count_tasks(model: dict) -> int:
    '''Recursively count Task-stencil shapes in the diagram.'''
    count = 0

    def walk(shapes):
        nonlocal count
        for shape in shapes or []:
            stencil_id = (shape.get("stencil") or {}).get("id", "")
            if stencil_id == "Task":
                count += 1
            walk(shape.get("childShapes"))

    walk(model.get("childShapes"))
    return count


def load_and_filter(cfg: Config) -> list[dict]:
    '''Stream every CSV under cfg.input_path, keep rows that pass all
    filters, and return a list of lightweight row dicts (not yet converted
    to the target schema).'''
    csv_files = sorted(Path(cfg.input_path).glob(cfg.csv_glob)) \
        if Path(cfg.input_path).is_dir() else [Path(cfg.input_path)]

    if not csv_files:
        raise FileNotFoundError(f"No CSV files found at {cfg.input_path} matching {cfg.csv_glob}")

    logger.info("Found %d CSV file(s) to scan.", len(csv_files))

    kept: list[dict] = []
    stats = {"scanned": 0, "bad_json": 0, "wrong_notation": 0,
              "wrong_language": 0, "excluded_variant": 0,
              "size_out_of_range": 0, "kept": 0}

    for csv_path in csv_files:
        try:
            reader = pd.read_csv(
                csv_path, sep=cfg.csv_sep, encoding=cfg.csv_encoding,
                usecols=lambda c: c in REQUIRED_COLUMNS,
                chunksize=cfg.chunksize, dtype=str, on_bad_lines="skip",
            )
        except Exception as exc:
            logger.warning("Skipping unreadable file %s: %s", csv_path.name, exc)
            continue

        for chunk in tqdm(reader, desc=f"Scanning {csv_path.name}", unit="chunk"):
            for _, row in chunk.iterrows():
                stats["scanned"] += 1
                name = row.get("Name") or ""
                description = row.get("Description") or ""

                if any(m in name.lower() for m in cfg.excluded_name_markers):
                    stats["excluded_variant"] += 1
                    continue

                model = _is_valid_bpmn_json(row.get("Model JSON"), cfg)
                if model is None:
                    stats["bad_json"] += 1
                    continue

                lang = _extract_language(model, name, description, cfg)
                if lang != cfg.target_language:
                    stats["wrong_language"] += 1
                    continue

                n_tasks = _count_tasks(model)
                if not (cfg.min_tasks <= n_tasks <= cfg.max_tasks):
                    stats["size_out_of_range"] += 1
                    continue

                kept.append({
                    "revision_id": row.get("Revision ID"),
                    "model_id": row.get("Model ID"),
                    "organization_id": row.get("Organization ID"),
                    "datetime": row.get("Datetime"),
                    "name": name,
                    "description": description,
                    "model": model,
                    "n_tasks": n_tasks,
                })
                stats["kept"] += 1

    stats["wrong_notation"] = stats["bad_json"]
    logger.info("Load/filter complete: %s", stats)
    return kept, stats


print("functions defined: _is_valid_bpmn_json, _extract_language, _count_tasks, load_and_filter")

✅ Cell 6b (patch) OK — _extract_language now verifies against actual text
functions defined: _is_valid_bpmn_json, _extract_language, _count_tasks, load_and_filter


In [27]:
# Cell - 4
def sample_processes(rows: list[dict], cfg: Config) -> list[dict]:
    rng = random.Random(cfg.random_seed)
    if len(rows) <= cfg.sample_size:
        logger.warning(
            "Only %d rows passed filtering (< requested sample_size=%d). "
            "Using all available rows.", len(rows), cfg.sample_size
        )
        sample = rows[:]
    else:
        sample = rng.sample(rows, cfg.sample_size)
    rng.shuffle(sample)
    logger.info("Sampled %d processes.", len(sample))
    return sample


print("function defined: sample_processes")

function defined: sample_processes


In [29]:
# Cell - 5
import copy

# Temporarily point at just the single file for a fast first test
test_cfg = copy.deepcopy(CONFIG)
test_cfg.input_path = test_file  # the single 0.csv file, not the whole folder

rows, load_stats = load_and_filter(test_cfg)


print("Rows returned:", len(rows))
print("Stats:", load_stats)
if rows:
    print("\nFirst kept row preview:")
    print("  name:", rows[0]["name"])
    print("  n_tasks:", rows[0]["n_tasks"])

02:34:26 | INFO     | Found 1 CSV file(s) to scan.


Scanning 0.csv: 0chunk [00:00, ?chunk/s]

02:34:56 | INFO     | Load/filter complete: {'scanned': 10000, 'bad_json': 4030, 'wrong_notation': 4030, 'wrong_language': 3808, 'excluded_variant': 15, 'size_out_of_range': 92, 'kept': 2055}


Rows returned: 2055
Stats: {'scanned': 10000, 'bad_json': 4030, 'wrong_notation': 4030, 'wrong_language': 3808, 'excluded_variant': 15, 'size_out_of_range': 92, 'kept': 2055}

First kept row preview:
  name: Receipt of Application
  n_tasks: 7


In [31]:
import copy

quick_cfg = copy.deepcopy(CONFIG)
quick_cfg.sample_size = 5

sample = sample_processes(rows, quick_cfg)

print("Sample size returned:", len(sample))
for i, r in enumerate(sample):
    print(f"  [{i}] name={r['name']!r}, n_tasks={r['n_tasks']}, model_id={r['model_id']}")

02:35:51 | INFO     | Sampled 5 processes.


Sample size returned: 5
  [0] name='4.1.7 Review distribution planning policies', n_tasks=2, model_id=1cc78dd7432b4712bbbf715d0a46c76b
  [1] name='SRSCH Revision (SOLL)', n_tasks=36, model_id=1ce01729ac7848b79702d4aa412ca85b
  [2] name='Procurement of Work Equipment', n_tasks=5, model_id=1d07b4f82a38424e9eda4d3956817215
  [3] name='Procurement of Work Equipment', n_tasks=5, model_id=1c357aa5a12c431ebbd817b3b6d2282b
  [4] name='Playground', n_tasks=3, model_id=1bc69e80d2444f4099579ec4d7ab946b


In [32]:
# Cell - 7 
def _walk_all_shapes(shapes, parent_type=None):
    '''Yield (shape, parent_type) for every shape in the tree, depth-first.'''
    for shape in shapes or []:
        stencil_id = (shape.get("stencil") or {}).get("id", "")
        yield shape, stencil_id
        yield from _walk_all_shapes(shape.get("childShapes"), stencil_id)


def _extract_flow_graph(model: dict) -> dict:
    '''Extract tasks, events, gateways and sequence flows from a Signavio
    BPMN JSON tree into flat lists, preserving resourceId linkage so we can
    rebuild ordering and gateway branches.'''
    tasks, gateways, events, flows = [], [], [], []

    for shape, stencil_id in _walk_all_shapes(model.get("childShapes")):
        rid = shape.get("resourceId")
        name = (shape.get("properties") or {}).get("name", "") or ""
        name = re.sub(r"\s+", " ", name).strip()
        outgoing = [o.get("resourceId") for o in shape.get("outgoing", [])]

        if stencil_id == "Task":
            tasks.append({"id": rid, "name": name or "Untitled Task", "outgoing": outgoing})
        elif "Gateway" in stencil_id:
            gateways.append({"id": rid, "name": name, "stencil": stencil_id, "outgoing": outgoing})
        elif "Event" in stencil_id:
            events.append({"id": rid, "name": name, "stencil": stencil_id, "outgoing": outgoing})
        elif stencil_id == "SequenceFlow":
            target = (shape.get("target") or {}).get("resourceId")
            flows.append({
                "id": rid,
                "name": (shape.get("properties") or {}).get("name", ""),
                "source_hint": None,
                "target": target,
            })

    return {"tasks": tasks, "gateways": gateways, "events": events, "flows": flows}


def _build_minimal_bpmn_xml(flow_graph: dict, process_name: str, process_code: str) -> str:
    '''Build a minimal, valid BPMN 2.0 XML document from the extracted
    flow graph. Not a full re-export of the original diagram -- a compact
    structural approximation sufficient for downstream parsing.'''
    ET.register_namespace("bpmn", "http://www.omg.org/spec/BPMN/20100524/MODEL")
    NS = "http://www.omg.org/spec/BPMN/20100524/MODEL"
    definitions = ET.Element(f"{{{NS}}}definitions", {
        "id": f"Definitions_{process_code}",
        "targetNamespace": "http://synthetic.local/bpmn",
    })
    process_el = ET.SubElement(definitions, f"{{{NS}}}process", {
        "id": f"Process_{process_code}", "name": process_name, "isExecutable": "false",
    })

    for t in flow_graph["tasks"]:
        ET.SubElement(process_el, f"{{{NS}}}task", {"id": t["id"], "name": t["name"]})
    for e in flow_graph["events"]:
        tag = "startEvent" if "Start" in e["stencil"] else "endEvent" if "End" in e["stencil"] else "intermediateCatchEvent"
        ET.SubElement(process_el, f"{{{NS}}}{tag}", {"id": e["id"], "name": e.get("name", "")})
    for g in flow_graph["gateways"]:
        tag = ("exclusiveGateway" if "Exclusive" in g["stencil"]
               else "parallelGateway" if "Parallel" in g["stencil"]
               else "inclusiveGateway" if "Inclusive" in g["stencil"]
               else "eventBasedGateway")
        ET.SubElement(process_el, f"{{{NS}}}{tag}", {"id": g["id"], "name": g.get("name", "")})
    for f in flow_graph["flows"]:
        if f["target"]:
            ET.SubElement(process_el, f"{{{NS}}}sequenceFlow", {
                "id": f["id"], "targetRef": f["target"],
                **({"name": f["name"]} if f["name"] else {}),
            })

    xml_bytes = ET.tostring(definitions, encoding="utf-8", xml_declaration=True)
    return xml_bytes.decode("utf-8")


def _synth_job(rng: random.Random, cfg: Config, job_id: int) -> dict:
    title = rng.choice(cfg.default_job_titles)
    return {
        "job_id": job_id,
        "jobCode": f"SYN-J-{job_id}",
        "job_level_id": rng.randint(1, 6),
        "hourlyRate": rng.randint(*cfg.hourly_rate_range),
        "maxHoursPerDay": 8,
        "description": f"Synthetic role: {title}",
        "name": title,
        "capacity_buffer": str(rng.choice([5, 10, 15, 20])),
        "days_per_week": "5",
        "hours_per_day": "8",
        "currencyType": cfg.default_currency,
    }


def _synth_gateways(flow_graph: dict, rng: random.Random) -> list[dict]:
    result = []
    for i, g in enumerate(flow_graph["gateways"], start=1):
        gtype = ("EXCLUSIVE" if "Exclusive" in g["stencil"]
                 else "PARALLEL" if "Parallel" in g["stencil"]
                 else "INCLUSIVE" if "Inclusive" in g["stencil"] else "EVENT_BASED")
        n_branches = max(2, len(g["outgoing"]))
        raw_probs = [rng.random() + 0.1 for _ in range(n_branches)]
        total = sum(raw_probs)
        probs = [round(p / total, 2) for p in raw_probs]

        branches = [{
            "id": 10_000 + i * 10 + b,
            "gateway_pk_id": 1000 + i,
            "is_default": b == 0,
            "target_task_id": None,
            "condition": g["name"] or f"branch_{b+1}",
            "end_event_name": None,
            "end_task_id": None,
            "connect_to_end": rng.random() < 0.3,
            "target_gateway_id": None,
            "probability": probs[b],
        } for b in range(n_branches)]

        result.append({
            "gateway_pk_id": 1000 + i,
            "gateway_type": gtype,
            "after_task_id": None,
            "name": g["name"] or f"Gateway {i}",
            "converge_at_task_id": None,
            "converge_gateway_name": "",
            "converge_to_end": False,
            "converge_at_gateway_id": None,
            "after_gateway_id": None,
            "branches": branches,
        })
    return result


def convert_to_schema(row: dict, process_id: int, cfg: Config) -> dict:
    '''Convert one filtered raw row into the 1972.json-style schema.
    Raises ValueError on unrecoverable structural problems so the caller
    can log-and-skip without corrupting the output set.'''
    model = row["model"]
    flow_graph = _extract_flow_graph(model)

    if not flow_graph["tasks"]:
        raise ValueError("No tasks extracted from diagram")

    rng = random.Random(cfg.synth_seed ^ process_id)
    process_code = f"SYN-P-{process_id}"
    bpmn_xml = _build_minimal_bpmn_xml(flow_graph, row["name"] or process_code, process_code)

    process_tasks = []
    job_id_counter = 1
    for order, t in enumerate(flow_graph["tasks"], start=1):
        proc_time = rng.randint(*cfg.process_time_range_min)
        rework_frac = round(rng.uniform(*cfg.rework_time_fraction_range), 2)
        n_jobs = rng.choice([1, 1, 1, 2])
        job_tasks = []
        for _ in range(n_jobs):
            job = _synth_job(rng, cfg, job_id_counter)
            job_tasks.append({
                "job_id": job["job_id"],
                "task_id": 5000 + order,
                "role": rng.choice(["R", "A", "C", "I"]),
                "time_allocation_percentage": round(rng.uniform(1, 20), 2),
                "job": job,
            })
            job_id_counter += 1

        process_tasks.append({
            "process_task_id": 6000 + order,
            "process_id": process_id,
            "task_id": 5000 + order,
            "order": order,
            "child_process_id": None,
            "value_classification": rng.choice(["VA", "BVA", "NVA"]),
            "value_rationale": None,
            "bva_business_goal": None,
            "value_source": "synthetic",
            "task": {
                "task_id": 5000 + order,
                "task_code": f"SYN-T-{process_id}-{order}",
                "task_company_id": None,
                "task_name": t["name"],
                "task_overview": f"Synthetically enriched task extracted from diagram element {t['id']}.",
                "status_id": 1,
                "task_version": 0,
                "expected_process_time": proc_time,
                "expected_rework_time": round(proc_time * rework_frac),
                "expected_waiting_time": rng.choice([None, rng.randint(1, 30)]),
                "frequency_interval": 1,
                "frequency_period": rng.choice(["DAY", "WEEK", "MONTH"]),
                "occurrences": "1",
                "jobTasks": job_tasks,
            },
            "child_process": None,
        })

    total_time = sum(pt["task"]["expected_process_time"] for pt in process_tasks)

    return {
        "process_id": process_id,
        "company_id": 900_000 + process_id,
        "created_at": row["datetime"],
        "updated_at": row["datetime"],
        "capacity_requirement_minutes": total_time,
        "parent_process_id": None,
        "parent_task_id": None,
        "process_code": process_code,
        "process_name": row["name"] or process_code,
        "process_overview": row["description"] or "<p>No description provided in source data.</p>",
        "process_category_id": cfg.default_process_category_id,
        "process_status_id": 1,
        "process_version": 0,
        "bpmn_xml": bpmn_xml,
        "created_by": None,
        "updated_by": None,
        "PROCESS_STATUS": "CREATED",
        "bpmn_xml_updated_at": row["datetime"],
        "company": {
            "company_id": 900_000 + process_id,
            "companyCode": f"SYN-{process_id}",
            "name": cfg.default_org_name,
            "created_by": None,
            "org_type_id": 1,
        },
        "process": None,
        "creator": {"user_id": None, "name": "Synthetic Pipeline"},
        "processCategory": {
            "id": cfg.default_process_category_id,
            "description": "Auto-assigned category for synthetic dataset",
            "name": cfg.default_process_category_name,
        },
        "gateways": _synth_gateways(flow_graph, rng),
        "process_task": process_tasks,
        "_source": {
            "revision_id": row["revision_id"],
            "model_id": row["model_id"],
            "organization_id": row["organization_id"],
        },
    }


print("✅ Cell 7 OK — functions defined: _extract_flow_graph, _build_minimal_bpmn_xml, _synth_job, _synth_gateways, convert_to_schema")

✅ Cell 7 OK — functions defined: _extract_flow_graph, _build_minimal_bpmn_xml, _synth_job, _synth_gateways, convert_to_schema


In [35]:
# Cell - 8 
REQUIRED_TOP_LEVEL = [
    "process_id", "process_code", "process_name", "bpmn_xml",
    "gateways", "process_task",
]


def validate_record(record: dict) -> list[str]:
    '''Return a list of validation problems (empty list == valid).'''
    problems = []

    for key in REQUIRED_TOP_LEVEL:
        if key not in record or record[key] in (None, ""):
            problems.append(f"missing/empty field: {key}")

    if not record.get("process_task"):
        problems.append("process_task list is empty")
    else:
        for pt in record["process_task"]:
            task = pt.get("task", {})
            if task.get("expected_process_time", 0) <= 0:
                problems.append(f"task {task.get('task_code')} has non-positive process time")
            if not task.get("jobTasks"):
                problems.append(f"task {task.get('task_code')} has no job assignments")

    for gw in record.get("gateways", []):
        probs = [b["probability"] for b in gw.get("branches", [])]
        if probs and abs(sum(probs) - 1.0) > 0.05:
            problems.append(f"gateway {gw.get('name')} branch probabilities sum to {sum(probs):.2f}, not ~1.0")

    try:
        ET.fromstring(record["bpmn_xml"])
    except ET.ParseError as exc:
        problems.append(f"invalid bpmn_xml: {exc}")

    return problems


print("✅ Cell 8 OK — function defined: validate_record")

✅ Cell 8 OK — function defined: validate_record


In [37]:
# Cell - 9
def convert_and_validate_all(sample: list[dict], cfg: Config) -> tuple[list[dict], dict]:
    converted = []
    conversion_errors = []
    validation_errors = []

    for i, row in enumerate(tqdm(sample, desc="Converting to schema", unit="proc")):
        process_id = 100_000 + i
        try:
            record = convert_to_schema(row, process_id, cfg)
        except Exception as exc:
            conversion_errors.append({"index": i, "name": row.get("name"), "error": str(exc)})
            continue

        problems = validate_record(record)
        if problems:
            validation_errors.append({"process_id": process_id, "problems": problems})
            continue

        converted.append(record)

    stats = {
        "attempted": len(sample),
        "converted_ok": len(converted),
        "conversion_failures": len(conversion_errors),
        "validation_failures": len(validation_errors),
        "conversion_error_samples": conversion_errors[:10],
        "validation_error_samples": validation_errors[:10],
    }
    logger.info(
        "Conversion complete: %d/%d succeeded (%d conversion errors, %d validation failures).",
        stats["converted_ok"], stats["attempted"],
        stats["conversion_failures"], stats["validation_failures"],
    )
    return converted, stats


print("✅ Cell 9 OK — function defined: convert_and_validate_all")

✅ Cell 9 OK — function defined: convert_and_validate_all


In [39]:
# Cell - 10
def split_and_save(records: list[dict], cfg: Config) -> dict:
    rng = random.Random(cfg.random_seed)
    shuffled = records[:]
    rng.shuffle(shuffled)

    split_idx = round(len(shuffled) * cfg.train_ratio)
    train_records, eval_records = shuffled[:split_idx], shuffled[split_idx:]

    manifest = {"train": [], "eval": []}

    for split_name, split_records in (("train", train_records), ("eval", eval_records)):
        out_dir = cfg.output_path / getattr(cfg, f"{split_name}_dirname")
        for record in tqdm(split_records, desc=f"Saving {split_name}", unit="file"):
            filename = f"{record['process_code']}.json"
            out_path = out_dir / filename
            try:
                with open(out_path, "w", encoding="utf-8") as f:
                    json.dump(record, f, indent=2, ensure_ascii=False)
                manifest[split_name].append(filename)
            except OSError as exc:
                logger.error("Failed to write %s: %s", out_path, exc)

    metadata = {
        "config": {k: (str(v) if isinstance(v, Path) else v)
                   for k, v in vars(cfg).items()},
        "counts": {
            "train": len(manifest["train"]),
            "eval": len(manifest["eval"]),
            "total": len(manifest["train"]) + len(manifest["eval"]),
        },
        "manifest": manifest,
        "generated_at": pd.Timestamp.now(tz="UTC").isoformat(),
    }

    meta_path = cfg.output_path / cfg.metadata_filename
    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2)

    logger.info("Saved %d train / %d eval records. Metadata: %s",
                metadata["counts"]["train"], metadata["counts"]["eval"], meta_path)
    return metadata


print("✅ Cell 10 OK — function defined: split_and_save")

✅ Cell 10 OK — function defined: split_and_save


In [44]:
# Cell - 11
def build_stats_report(load_stats: dict, conv_stats: dict, split_meta: dict,
                        converted_records: list[dict], cfg: Config) -> dict:
    n_tasks_per_process = [len(r["process_task"]) for r in converted_records]
    n_gateways_per_process = [len(r["gateways"]) for r in converted_records]
    proc_times = [pt["task"]["expected_process_time"]
                  for r in converted_records for pt in r["process_task"]]

    def _avg(vals):
        return round(sum(vals) / len(vals), 2) if vals else 0

    report = {
        "stage_1_load_and_filter": load_stats,
        "stage_2_conversion": {
            k: v for k, v in conv_stats.items() if not k.endswith("_samples")
        },
        "stage_3_split": split_meta["counts"],
        "dataset_characteristics": {
            "avg_tasks_per_process": _avg(n_tasks_per_process),
            "min_tasks_per_process": min(n_tasks_per_process, default=0),
            "max_tasks_per_process": max(n_tasks_per_process, default=0),
            "avg_gateways_per_process": _avg(n_gateways_per_process),
            "avg_task_process_time_minutes": _avg(proc_times),
        },
        "sample_conversion_errors": conv_stats.get("conversion_error_samples", []),
        "sample_validation_errors": conv_stats.get("validation_error_samples", []),
    }

    stats_path = cfg.output_path / cfg.stats_filename
    with open(stats_path, "w", encoding="utf-8") as f:
        json.dump(report, f, indent=2)

    print("=" * 60)
    print("PIPELINE SUMMARY")
    print("=" * 60)
    print(f"Rows scanned:              {load_stats['scanned']}")
    print(f"Passed BPMN/language/size filters: {load_stats['kept']}")
    print(f"Sampled for conversion:    {conv_stats['attempted']}")
    print(f"Successfully converted:    {conv_stats['converted_ok']}")
    print(f"  Conversion failures:     {conv_stats['conversion_failures']}")
    print(f"  Validation failures:     {conv_stats['validation_failures']}")
    print(f"Final train / eval split:  {split_meta['counts']['train']} / {split_meta['counts']['eval']}")
    print(f"Avg tasks per process:     {report['dataset_characteristics']['avg_tasks_per_process']}")
    print(f"Avg gateways per process:  {report['dataset_characteristics']['avg_gateways_per_process']}")
    print(f"Avg task time (min):       {report['dataset_characteristics']['avg_task_process_time_minutes']}")
    print(f"\nFull report written to: {stats_path}")
    print("=" * 60)

    return report


print("✅ Cell 11 OK — function defined: build_stats_report")

✅ Cell 11 OK — function defined: build_stats_report


In [47]:
# Cell - 12
def run_pipeline(cfg: Config = CONFIG) -> dict:
    try:
        rows, load_stats = load_and_filter(cfg)
    except FileNotFoundError as exc:
        logger.error("Pipeline aborted at load stage: %s", exc)
        raise

    if not rows:
        logger.error("No rows passed filtering — check csv_sep / paths / filters in Config.")
        return {}

    sample = sample_processes(rows, cfg)
    converted, conv_stats = convert_and_validate_all(sample, cfg)

    if not converted:
        logger.error("No records survived conversion/validation — aborting before save.")
        return {}

    split_meta = split_and_save(converted, cfg)
    report = build_stats_report(load_stats, conv_stats, split_meta, converted, cfg)
    return report


print("✅ Cell 12 OK — function defined: run_pipeline")

✅ Cell 12 OK — function defined: run_pipeline
